<a href="https://colab.research.google.com/github/333universe/npkstep2/blob/main/npkmath_complete_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Complete NPKmath Evaluation Pipeline
Welcome! This notebook combines **Step 1** (Setup & Engine Testing) and **Step 2** (AI Benchmarking) into a single, straightforward pipeline.

### ⚠️ Crucial First Step:
Before running anything, look at the left sidebar in Colab, click the **Folder icon (📁)**, and click the **Upload icon**. Upload both of your zip files here:
1. `npkmath_step1.zip`
2. `npkmath_step2.zip`

## 🛠️ Cell 1: Environment Setup & Package Extraction
This cell automatically finds your uploaded zip files, unzips them safely, and installs the required mathematical frameworks.

In [1]:
import os, glob
if not os.path.exists('npkmath'):
    # Find the uploaded step zip files, prioritize step2 if available
    zips = sorted(glob.glob('/content/npkmath_step*.zip'))
    if not zips:
        print("❌ ERROR: Please upload 'npkmath_step1.zip' or 'npkmath_step2.zip' to the /content folder first!")
    else:
        z = zips[-1]
        folder = os.path.basename(z)[:-4]
        print(f"📦 Unzipping framework from: {z}")
        !unzip -oq {z} -d /content
        %cd /content/{folder}

!pip -q install -e ".[dev]"

📦 Unzipping framework from: /content/npkmath_step2.zip
/content/npkmath_step2
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for npkmath (pyproject.toml) ... done


## 🧪 Cell 2: Run Internal Unit Tests
This runs the self-checks to ensure the mathematical framework is properly installed and behaving correctly. You should see a green success status or a simple test summary.

In [18]:
!python -m pytest -q

...............................................................          [100%]
63 passed in 34.57s


## 📈 Cell 3: Local Engine Benchmarking
Runs a performance check on the internal engine to verify math processing speeds.

In [19]:
!python -m npkmath.benchmark

NPKmath verifier benchmark
--------------------------------------------
              claims: 53
                full: 51
             partial: 2
                miss: 0
     critical_errors: 0
     false_assurance: 0
          exact_rate: 0.962
    precision_PROVED: 1.0
   precision_REFUTED: 1.0
              max_ms: 393.459

Not a full match:
  [        partial] eq-sqrt-abs                  truth=TRUE        got=SUPPORTED
  [        partial] eq-log-exp-real              truth=TRUE        got=SUPPORTED

Saved benchmark_report.json


## 📝 Cell 4: Initialize Dataset & Offline Evaluation Builder
This cell generates the evaluation dataset consisting of 60 unique math problems and establishes our testing context.

In [20]:
import json
from npkmath.dataset import build_dataset, majority_baseline
from npkmath.engine import NPKCoreEngine
from npkmath.llm_eval import *
from npkmath.signals import AIFeedbackInterface

engine = NPKCoreEngine()
items = build_dataset(seed=0, max_items=60)
cb = ContextBuilder(items, engine) # verifies every claim once, builds related context
print(f"📊 Created {len(items)} questions.")
print('Always-guess-the-commonest floor baseline score =', majority_baseline(items))

📊 Created 60 questions.
Always-guess-the-commonest floor baseline score = {'label': 'FALSE', 'accuracy': 0.483, 'counts': {'FALSE': 29, 'TRUE': 25, 'CONDITIONAL': 6}}


## 📑 Cell 5: Export Evaluation Prompts (Copy-Paste Driver)
Change the parameters `WHICH` and `N` in this cell to fetch different prompts.
*   To get **Baseline** prompts: Set `WHICH = base_batches`
*   To get **Grounded** prompts: Set `WHICH = grounded_batches`
*   Cycle `N` through **0, 1, and 2** to print out each of your three evaluation batches.

In [33]:
import importlib
import npkmath.llm_eval as m
importlib.reload(m)
from npkmath.llm_eval import *

base_batches = export_batches(items, None, 20)
grounded_batches = export_batches(items, cb.contexts(), 20)
print(f"Generated {len(base_batches)} batches per testing condition.")

# --- CONFIGURATION AREA: CHANGE THESE TO GET PROGRESSIVE PROMPTS ---
WHICH = base_batches      # Choose: base_batches OR grounded_batches
N = 0                         # Choose Batch Index: 0, 1, or 2
# ------------------------------------------------------------------

print(f"\n--- Showing Prompt for Condition: {'Baseline' if WHICH is base_batches else 'Grounded'} | Batch: {N} ---\n")
print(WHICH[N][0])

Generated 3 batches per testing condition.

--- Showing Prompt for Condition: Baseline | Batch: 0 ---

Treat all variables and symbols as real numbers. Ignore isolated exceptional points (such as division by zero). If a side is not a real number over a whole range of values (for example log or sqrt of a negative number) the claim fails there.
Answer TRUE if the statement holds for every real value. Answer CONDITIONAL if it holds for positive values but fails for some negative values. Answer FALSE otherwise.

For each numbered claim answer with one word. Reply only with lines in the form 'Q01: TRUE'.

Q01: exp(x)**2 = exp(x**3)
Q02: sin(2*x) = 2*cos(x)*cos(x)
Q03: A = [[a, b], [c, d]]. Statement: A^T = A
Q04: 1-2*sin(x)**2 = cos(2*x)
Q05: sin(2*x) = 2*sin(x)*cos(x)
Q06: x**3-y**3 = (x-y)*(x**2-x*y+y**2)
Q07: sin(x)+sin(y) = sin(x+y)
Q08: exp(I*x) = I*sin(x)+cos(x)
Q09: (x**2-1)/(x-1) = x+2
Q10: x**3+3*x**2*y+3*x*y**2+y**3 = (x+y)**3
Q11: sin(x+y) = sin(x)*cos(y)+sin(y)*cos(x)
Q12: (x+y)

In [22]:
import inspect
from npkmath.llm_eval import ContextBuilder
print(inspect.getsource(ContextBuilder))

class ContextBuilder:
    """Verifies every item once with the engine, then builds per-target context."""

    def __init__(self, items: Sequence[Item], engine: Optional[NPKCoreEngine] = None,
                 k: int = 4, seed: int = 0, exclude_family: bool = True,
                 source_reliability: float = 1.0):
        self.items, self.k, self.seed, self.exclude_family = list(items), k, seed, exclude_family
        self.engine = engine or NPKCoreEngine()
        self._fb = AIFeedbackInterface()
        self.signals: Dict[str, VerificationSignal] = {}
        for it in self.items:
            if it.kind == "equality":
                sig = verify_equality_and_emit(self.engine, self._fb, it.id, it.payload["lhs"],
                                               it.payload["rhs"], source_reliability)
            else:
                sig = verify_matrix_and_emit(self.engine, self._fb, it.id, it.payload["op"],
                                             it.payload["a"], it.payload["b"],

In [23]:
import npkmath.llm_eval
print(dir(npkmath.llm_eval))

['AIFeedbackInterface', 'ANSWERS', 'Callable', 'ContextBuilder', 'Dict', 'Item', 'LABEL', 'List', 'MockModel', 'NPKCoreEngine', 'Optional', 'RULES', 'Sequence', 'Tuple', 'VerificationSignal', '_FUNCS', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__spec__', '_sign_test_p', '_tokens', 'accuracy', 'answer_with_verification', 'collect_answers', 'compact_context', 'compare', 'export_batches', 'log_feedback', 'make_batch_prompt', 'make_prompt', 'math', 'parse_answer', 'parse_batch', 'random', 're', 'score', 'summarize', 'verify_equality_and_emit', 'verify_matrix_and_emit']


## 📥 Cell 6: Collect Model Answers
Paste the text output answers directly between the triple quotes below for each condition/batch combination.

In [46]:
baseline_replies = {
 0: '''Q01: FALSE
Q02: FALSE
Q03: FALSE
Q04: TRUE
Q05: TRUE
Q06: FALSE
Q07: FALSE
Q08: TRUE
Q09: FALSE
Q10: TRUE
Q11: TRUE
Q12: FALSE
Q13: TRUE
Q14: TRUE
Q15: TRUE
Q16: FALSE
Q17: CONDITIONAL
Q18: TRUE
Q19: TRUE
Q20: FALSE''',
 1: '''Q01: FALSE
Q02: FALSE
Q03: TRUE
Q04: FALSE
Q05: FALSE
Q06: TRUE
Q07: CONDITIONAL
Q08: FALSE
Q09: FALSE
Q10: FALSE
Q11: CONDITIONAL
Q12: FALSE
Q13: TRUE
Q14: FALSE
Q15: TRUE
Q16: CONDITIONAL
Q17: CONDITIONAL
Q18: TRUE
Q19: FALSE
Q20: FALSE''',
 2: '''Q01: TRUE
Q02: TRUE
Q03: FALSE
Q04: FALSE
Q05: FALSE
Q06: CONDITIONAL
Q07: TRUE
Q08: CONDITIONAL
Q09: TRUE
Q10: FALSE
Q11: FALSE
Q12: TRUE
Q13: TRUE
Q14: TRUE
Q15: FALSE
Q16: FALSE
Q17: TRUE
Q18: TRUE
Q19: FALSE
Q20: TRUE''',
}

grounded_replies = {
 0: '''Q01: FALSE
Q02: FALSE
Q03: FALSE
Q04: TRUE
Q05: TRUE
Q06: FALSE
Q07: FALSE
Q08: TRUE
Q09: FALSE
Q10: TRUE
Q11: TRUE
Q12: FALSE
Q13: TRUE
Q14: TRUE
Q15: TRUE
Q16: FALSE
Q17: CONDITIONAL
Q18: TRUE
Q19: TRUE
Q20: FALSE''',
 1: '''Q01: FALSE
Q02: FALSE
Q03: TRUE
Q04: FALSE
Q05: FALSE
Q06: TRUE
Q07: CONDITIONAL
Q08: FALSE
Q09: FALSE
Q10: FALSE
Q11: CONDITIONAL
Q12: FALSE
Q13: TRUE
Q14: FALSE
Q15: TRUE
Q16: CONDITIONAL
Q17: CONDITIONAL
Q18: TRUE
Q19: FALSE
Q20: FALSE''',
 2: '''Q01: TRUE
Q02: TRUE
Q03: FALSE
Q04: FALSE
Q05: FALSE
Q06: CONDITIONAL
Q07: TRUE
Q08: CONDITIONAL
Q09: TRUE
Q10: FALSE
Q11: FALSE
Q12: TRUE
Q13: TRUE
Q14: TRUE
Q15: FALSE
Q16: FALSE
Q17: TRUE
Q18: TRUE
Q19: FALSE
Q20: TRUE''',
}

## 📊 Cell 7: Process Answers & Calculate Metric Scores
Run this final cell after pasting the answers into Cell 6. It calculates metrics and exports `results_step2.json`.

In [47]:
def gather(batches, replies):
    answers = {}
    for k, (prompt, qmap) in enumerate(batches):
        answers.update(parse_batch(replies.get(k, ''), qmap))
    return answers

MODEL_NAME = 'my-tested-model'
fb = AIFeedbackInterface()
rows = {}

for cond, batches, replies in [('baseline', base_batches, baseline_replies),
                               ('grounded', grounded_batches, grounded_replies)]:
    ans = gather(batches, replies)
    if ans:
        answered = [it for it in items if it.id in ans]
        rows[cond] = score(answered if len(ans) < len(items) else items, ans, cond)
        log_feedback(fb, rows[cond], MODEL_NAME)

if rows:
    report = summarize(items, rows)
    print("\n===== FINAL BENCHMARK PERFORMANCE REPORT =====")
    print(json.dumps(report, indent=1))
    json.dump({'report': report, 'feedback_log': fb.feedback_log}, open('results_step2.json', 'w'), indent=1)
    print('\n✅ Successfully compiled and saved results_step2.json')
else:
    print('⚠️ Operational Notice: Paste your model text outputs into Cell 6 before executing evaluation metrics.')


===== FINAL BENCHMARK PERFORMANCE REPORT =====
{
 "n_items": 60,
 "majority_baseline": {
  "label": "FALSE",
  "accuracy": 0.483,
  "counts": {
   "FALSE": 29,
   "TRUE": 25,
   "CONDITIONAL": 6
  }
 },
 "conditions": {
  "baseline": {
   "accuracy": 0.983,
   "false_assurance": 0,
   "unparsed": 0
  },
  "grounded": {
   "accuracy": 0.983,
   "false_assurance": 0,
   "unparsed": 0
  }
 },
 "comparisons": {
  "baseline->grounded": {
   "n": 60,
   "baseline_accuracy": 0.983,
   "other_accuracy": 0.983,
   "gained": 0,
   "lost": 0,
   "net": 0,
   "sign_test_p": 1.0,
   "baseline_false_assurance": 0,
   "other_false_assurance": 0,
   "note": "p is a rough guide only; small samples and one model prove little."
  }
 }
}

✅ Successfully compiled and saved results_step2.json


In [48]:
import inspect
from npkmath.llm_eval import parse_batch
print(inspect.getsource(parse_batch))

def parse_batch(text: str, qmap: Dict[str, str]) -> Dict[str, str]:
    """Parse 'Q01: TRUE' style lines into {item_id: answer}. Unparseable lines are skipped."""
    out: Dict[str, str] = {}
    for m in re.finditer(r"\b(Q\d{2,3})\b\W{0,6}\**\s*(TRUE|FALSE|CONDITIONAL)\b", text or "", re.IGNORECASE):
        q = m.group(1).upper()
        if q in qmap:
            out[qmap[q]] = m.group(2).upper()
    return out

